# Simulation of Φ-sat-2 Bands from Sentinel-2 using SimulationPipeline

This notebook demonstrates the Φ-sat-2 simulation pipeline with intermediate visualizations. It processes pre-downloaded Sentinel-2 L1C .tiff files and applies configurable simulation steps to generate Φ-sat-2 imagery.

## Workflow Overview

The simulation pipeline performs the following steps:
1. **Radiance Conversion**: Convert reflectance to radiance using solar irradiance metadata
2. **Panchromatic Band**: Compute PAN as weighted combination of input bands
3. **Spatial Resampling**: Resample from 10m to 4.75m pixel size
4. **Band Misalignment**: Simulate L1A/L1B/L1C band-to-band shifts
5. **SNR Simulation**: Add signal degradation due to sensor noise
6. **PSF Filtering**: Simulate point spread function effects
7. **Reflectance Conversion**: Convert radiance back to reflectance (for L1C)

## Table of Contents
0. [Import Libraries](#imports)
1. [Configure Pipeline](#configure)
2. [Load S2 Data](#load-data)
3. [Radiance Conversion](#radiance)
4. [Add Panchromatic Band](#panchromatic)
5. [Spatial Resampling](#resampling)
6. [Band Misalignment](#misalignment)
7. [SNR Simulation](#snr)
8. [PSF Filtering](#psf)
9. [Reflectance Conversion](#reflectance)
10. [Result Visualization](#visualization)

<a name="imports"></a>
## 0. Import Required Libraries

In [4]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import warnings
warnings.filterwarnings('ignore')

import os
from pathlib import Path
import json
import numpy as np
import matplotlib.pyplot as plt
import rasterio
from tqdm import tqdm

from eolearn.core.eodata import EOPatch
from eolearn.core.constants import FeatureType
from eolearn.core.core_tasks import RemoveFeatureTask

from eolearn.features.utils import spatially_resize_image as resize_images
from eolearn.core.core_tasks import MapFeatureTask
from eolearn.io.raster_io import ExportToTiffTask

from terra_sat_drift.data_simulation.phisat2_constants import S2_RESOLUTION, PHISAT2_RESOLUTION, ProcessingLevels
from terra_sat_drift.data_simulation.phisat2_utils import (
    AddMetadataTask,
    CalculateRadianceTask,
    CalculateReflectanceTask,
    AddPANBandTask,
    BandMisalignmentTask,
    PhisatCalculationTask,
)

from terra_sat_drift.data_simulation.simulation_pipeline import SimulationPipeline
from terra_sat_drift.data_simulation.simulation_config import SimulationConfig, SimulationSteps

print("✓ All libraries imported successfully")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


ModuleNotFoundError: No module named 'simulation_config'

<a name="configure"></a>
## 1. Configure Simulation Pipeline

Set up the simulation configuration with desired steps enabled. This notebook uses the same configuration as Example 5 from simulation_examples.py.

In [2]:
# Define simulation steps (Example 5 configuration)
steps = SimulationSteps(
    radiance=True,
    add_panchromatic=True,
    band_misalignment=True,
    snr_simulation=True,
    psf_filtering=True,
    reflectance_conversion=True,
)

# Configure the pipeline
config = SimulationConfig(
    steps=steps,
    output_dir="tiff_folder/simulated_s2",
    processing_level="L1A",  # Options: "L1A", "L1B", "L1C"
    phisat2_exec_path="executables/phisat2_unix.bin",  # Set path to your OS-specific binary
    snr_psf_method="executable",  # Options: "executable" or "alternative"
    cell_size=256,
    grid_overlap=0.0,
)

# Create output directory
Path(config.output_dir).mkdir(parents=True, exist_ok=True)

# Print configuration
print("Simulation Configuration:")
print(f"  Processing Level: {config.processing_level}")
print(f"  SNR/PSF Method: {config.snr_psf_method}")
print(f"  Output Directory: {config.output_dir}")
print(f"\nEnabled Steps:")
for step, enabled in steps.as_dict().items():
    status = "✓" if enabled else "✗"
    print(f"  {status} {step.replace('_', ' ').title()}")

Simulation Configuration:
  Processing Level: L1A
  SNR/PSF Method: executable
  Output Directory: tiff_folder/simulated_s2

Enabled Steps:
  ✓ Radiance
  ✓ Add Panchromatic
  ✓ Band Misalignment
  ✓ Snr Simulation
  ✓ Psf Filtering
  ✓ Reflectance Conversion


<a name="load-data"></a>
## 2. Load and Prepare S2 Data

Load a single Sentinel-2 L1C .tiff file and create an EOPatch with the raw bands.

In [5]:
# Define paths
S2_SOURCE_DIR = Path("terra-sat-drift/tiff_folder/s2b_cropped")  # Directory containing S2 .tiff files
S2_FILES = list(S2_SOURCE_DIR.glob("*.tif"))

print(f"Found {len(S2_FILES)} S2 TIFF files in {S2_SOURCE_DIR}")
if S2_FILES:
    print(f"First file: {S2_FILES[0].name}")
    s2_file = S2_FILES[0]  # Use first file for this example
else:
    print("⚠ No .tif files found. Please adjust S2_SOURCE_DIR path.")

Found 0 S2 TIFF files in terra-sat-drift/tiff_folder/s2b_cropped
⚠ No .tif files found. Please adjust S2_SOURCE_DIR path.


In [ ]:
# Load S2 data from TIFF file
with rasterio.open(s2_file) as src:
    s2_data = src.read().astype(np.float32)  # Shape: (bands, height, width)
    
    # Get geospatial information
    from sentinelhub.geometry import BBox
    from sentinelhub.constants import CRS
    bounds = src.bounds
    bbox = BBox(
        bbox=(bounds.left, bounds.bottom, bounds.right, bounds.top),
        crs=CRS(src.crs)
    )

# Transpose from (bands, height, width) to (height, width, bands)
s2_data = np.transpose(s2_data, (1, 2, 0))
print(f"✓ Loaded S2 data with shape {s2_data.shape}")
print(f"  Data range: [{s2_data.min():.1f}, {s2_data.max():.1f}]")

# Create EOPatch with raw S2 bands
eopatch = EOPatch(bbox=bbox)
eopatch[FeatureType.DATA, "BANDS"] = s2_data[np.newaxis, :, :, :]  # Add time dimension

print(f"\n✓ Created EOPatch:")
print(f"  {eopatch}")

In [ ]:
# Visualize raw S2 reflectance (RGB composite: B04, B03, B02)
fig, ax = plt.subplots(figsize=(10, 10))
# Note: Band order depends on input TIFF structure
# Typical S2 order: B02(1), B03(2), B04(3), B08(4), B05(5), B06(6), B07(7)
rgb_idx = [2, 1, 0]  # Adjust based on your band order
rgb_composite = np.clip(2.5 * eopatch.data["BANDS"][0, :, :, rgb_idx], 0, 1)
ax.imshow(rgb_composite)
ax.set_title("Raw Sentinel-2 Reflectance (RGB)", fontsize=14)
ax.set_xlabel("Pixels")
ax.set_ylabel("Pixels")
plt.tight_layout()
plt.show()

<a name="radiance"></a>
## 3. Radiance Conversion

Convert reflectance values to radiances using solar irradiance and Earth-Sun distance metadata. This is the first simulation step.

In [ ]:
# Load metadata (solar irradiance and Earth-Sun distance)
def load_metadata_from_file(tiff_path):
    """Load metadata JSON file associated with a TIFF file."""
    tiff_path = Path(tiff_path)
    stem = tiff_path.stem
    parent_dir = tiff_path.parent
    
    # Try common naming patterns
    possible_paths = [
        parent_dir / f"{stem}_metadata.json",
        parent_dir / f"{stem}.json",
    ]
    
    for metadata_path in possible_paths:
        if metadata_path.exists():
            try:
                with open(metadata_path, 'r') as f:
                    return json.load(f)
            except Exception as e:
                print(f"Warning: Failed to load metadata: {e}")
    
    # If no metadata file found, create dummy metadata
    print("⚠ No metadata file found. Using default values.")
    return {
        "earth_sun_dist": 1.0,
        "solar_irradiances": {f"B{i:02d}": 1000 for i in range(2, 9)},
        "sun_zenith_angle": 30.0
    }

metadata = load_metadata_from_file(s2_file)
print(f"✓ Loaded metadata: {list(metadata.keys())}")

# Apply metadata to EOPatch
add_meta_task = AddMetadataTask()
try:
    eopatch = add_meta_task.execute(eopatch, metadata)
    print(f"✓ Applied metadata to EOPatch")
except Exception as e:
    print(f"⚠ Could not apply metadata task: {e}")
    # Manually add to metadata if needed
    for key, value in metadata.items():
        if isinstance(value, dict):
            for k, v in value.items():
                eopatch.meta_info[f"{key}_{k}"] = v
        else:
            eopatch.meta_info[key] = value

In [ ]:
# Apply Radiance Conversion
radiance_task = CalculateRadianceTask(
    (FeatureType.DATA, "BANDS"),
    (FeatureType.DATA, "BANDS-RAD")
)

if config.steps.radiance:
    eopatch = radiance_task.execute(eopatch)
    print("✓ Radiance conversion applied")
else:
    print("⊘ Radiance conversion skipped")

print(f"\nEOPatch contents:\n{eopatch}")

In [ ]:
# Visualize Reflectance vs Radiance comparison
if "BANDS-RAD" in eopatch.data:
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    # Reflectance
    axes[0].imshow(np.clip(2.5 * eopatch.data["BANDS"][0, :, :, rgb_idx], 0, 1))
    axes[0].set_title("Reflectance Values", fontsize=12)
    axes[0].set_xlabel("Pixels")
    axes[0].set_ylabel("Pixels")
    
    # Radiance
    axes[1].imshow(np.clip(eopatch.data["BANDS-RAD"][0, :, :, rgb_idx] / 100, 0, 1))
    axes[1].set_title("Radiance Values (normalized)", fontsize=12)
    axes[1].set_xlabel("Pixels")
    axes[1].set_ylabel("Pixels")
    
    plt.tight_layout()
    plt.show()

<a name="panchromatic"></a>
## 4. Add Panchromatic Band

Compute a pan-chromatic band as a weighted linear combination of the input bands.

In [ ]:
# Apply Add Panchromatic Band Task
input_feature = "BANDS-RAD" if "BANDS-RAD" in eopatch.data else "BANDS"

add_pan_task = AddPANBandTask(
    (FeatureType.DATA, input_feature),
    (FeatureType.DATA, "BANDS-RAD-PAN")
)

if config.steps.add_panchromatic:
    eopatch = add_pan_task.execute(eopatch)
    print("✓ Panchromatic band added")
    # The PAN band is inserted at index 3 (between B04 and B08)
    print(f"  New shape: {eopatch.data['BANDS-RAD-PAN'].shape}")
else:
    print("⊘ Panchromatic band generation skipped")

In [ ]:
# Visualize Panchromatic Band
if "BANDS-RAD-PAN" in eopatch.data:
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    
    # Original RGB
    axes[0].imshow(np.clip(2.5 * eopatch.data[input_feature][0, :, :, rgb_idx], 0, 1))
    axes[0].set_title("Original RGB", fontsize=12)
    axes[0].set_xlabel("Pixels")
    
    # Panchromatic band (at index 3)
    pan_band = eopatch.data["BANDS-RAD-PAN"][0, :, :, 3] / 100
    axes[1].imshow(pan_band, cmap="gray")
    axes[1].set_title("Panchromatic Band (PAN)", fontsize=12)
    axes[1].set_xlabel("Pixels")
    
    # PAN band histogram
    axes[2].hist(pan_band.ravel(), bins=50, edgecolor='black')
    axes[2].set_title("PAN Band Histogram", fontsize=12)
    axes[2].set_xlabel("Pixel Values")
    axes[2].set_ylabel("Frequency")
    
    plt.tight_layout()
    plt.show()

<a name="resampling"></a>
## 5. Spatial Resampling

Resample data from Sentinel-2 resolution (10m) to Φ-sat-2 resolution (4.75m).

In [ ]:
# Prepare features to resize
current_feature = "BANDS-RAD-PAN" if "BANDS-RAD-PAN" in eopatch.data else input_feature

features_to_resize = {
    FeatureType.DATA: [current_feature],
}

# Calculate new size based on resolution change
# S2: 10m pixels, Φ-sat-2: 4.75m pixels
original_size = eopatch.data[current_feature].shape[2:4]
scale_factor = S2_RESOLUTION / PHISAT2_RESOLUTION
new_size = (
    int(original_size[0] * scale_factor),
    int(original_size[1] * scale_factor)
)

print(f"Original size: {original_size}")
print(f"Scale factor: {scale_factor:.2f}")
print(f"New size (Φ-sat-2): {new_size}")

# Apply resizing tasks
resize_tasks = []
for feature_type in features_to_resize.keys():
    for feature in features_to_resize[feature_type]:
        resize_task = MapFeatureTask(
            (feature_type, feature),
            (feature_type, f"{feature}_RES"),
            resize_images,
            new_size=new_size,
            resize_method="nearest",
        )
        eopatch = resize_task.execute(eopatch)
        resize_tasks.append(resize_task)

print(f"✓ Spatial resampling applied")
current_feature = f"{current_feature}_RES"
print(f"  New feature: {current_feature}")
print(f"  New data shape: {eopatch.data[current_feature].shape}")

In [ ]:
# Visualize resampling effect
if current_feature in eopatch.data:
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    # Extract original and resampled
    original_data = eopatch.data[current_feature.replace("_RES", "")][0]
    resampled_data = eopatch.data[current_feature][0]
    
    # Display small region for detail
    region = slice(100, 200), slice(100, 200)
    
    im1 = axes[0].imshow(original_data[region, rgb_idx] / 100)
    axes[0].set_title(f"Original ({original_data.shape[:2]})", fontsize=12)
    axes[0].set_xlabel("Pixels")
    
    im2 = axes[1].imshow(resampled_data[region, rgb_idx] / 100)
    axes[1].set_title(f"Resampled ({resampled_data.shape[:2]})", fontsize=12)
    axes[1].set_xlabel("Pixels")
    
    plt.tight_layout()
    plt.show()

<a name="misalignment"></a>
## 6. Band Misalignment

Simulate band-to-band misalignment effects. The amount and pattern of misalignment depends on the processing level (L1A, L1B, or L1C).

In [ ]:
# Apply Band Misalignment
import cv2

processing_level = ProcessingLevels[config.processing_level]

band_misalign_task = BandMisalignmentTask(
    (FeatureType.DATA, current_feature),
    (FeatureType.DATA, "BANDS_MISALIGNED"),
    processing_level=processing_level,
    std_sea=6,
    interpolation_method=cv2.INTER_NEAREST,
)

if config.steps.band_misalignment:
    eopatch = band_misalign_task.execute(eopatch)
    print(f"✓ Band misalignment ({config.processing_level}) applied")
    
    # Show the shifts that were applied
    if "Shifts" in eopatch.meta_info:
        print(f"\n  Applied shifts (rounded to nearest pixel):")
        for band_idx, shifts in enumerate(eopatch.meta_info["Shifts"].values()):
            shift_mag = np.round(np.linalg.norm(shifts))
            print(f"    Band {band_idx}: {shift_mag} pixels")
else:
    print("⊘ Band misalignment skipped")
    
current_feature = "BANDS_MISALIGNED"

In [ ]:
# Visualize misalignment effects on individual bands
if "BANDS_MISALIGNED" in eopatch.data:
    original_feature = current_feature.replace("_MISALIGNED", "")
    
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    region = slice(50, 150), slice(50, 150)
    
    for band_idx, title in [(0, "Band 0"), (2, "Band 2"), (4, "Band 4"), (6, "Band 6")]:
        ax = axes[band_idx // 2, band_idx % 2]
        
        original = eopatch.data[original_feature][0, region[0], region[1], band_idx]
        misaligned = eopatch.data["BANDS_MISALIGNED"][0, region[0], region[1], band_idx]
        
        # Show difference
        diff = np.abs(original - misaligned)
        im = ax.imshow(diff, cmap="hot")
        ax.set_title(f"{title} - Misalignment Difference", fontsize=11)
        ax.set_xlabel("Pixels")
        plt.colorbar(im, ax=ax, label="Difference")
    
    plt.tight_layout()
    plt.show()

<a name="snr"></a>
## 7. SNR Simulation

Apply signal degradation due to Signal-to-Noise Ratio (SNR) specifications of the Φ-sat-2 sensor.

In [ ]:
# Apply SNR Simulation
snr_task = PhisatCalculationTask(
    input_feature=(FeatureType.DATA, current_feature),
    output_feature=(FeatureType.DATA, "BANDS_SNR"),
    executable=config.phisat2_exec_path,
    calculation="SNR",
)

if config.steps.snr_simulation:
    try:
        eopatch = snr_task.execute(eopatch)
        print("✓ SNR simulation applied")
    except Exception as e:
        print(f"⚠ SNR simulation failed: {e}")
        print("  (Check that phisat2 executable path is correct)")
        eopatch[FeatureType.DATA, "BANDS_SNR"] = eopatch[FeatureType.DATA, current_feature]
else:
    print("⊘ SNR simulation skipped")
    eopatch[FeatureType.DATA, "BANDS_SNR"] = eopatch[FeatureType.DATA, current_feature]

current_feature = "BANDS_SNR"

In [ ]:
# Visualize SNR effects (before/after noise)
if "BANDS_SNR" in eopatch.data:
    before_snr = eopatch.data["BANDS_MISALIGNED"][0]
    after_snr = eopatch.data["BANDS_SNR"][0]
    
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    region = slice(100, 200), slice(100, 200)
    
    # RGB comparison
    axes[0, 0].imshow(np.clip(before_snr[region, rgb_idx] / 100, 0, 1))
    axes[0, 0].set_title("Before SNR", fontsize=11)
    axes[0, 0].set_xlabel("Pixels")
    
    axes[0, 1].imshow(np.clip(after_snr[region, rgb_idx] / 100, 0, 1))
    axes[0, 1].set_title("After SNR", fontsize=11)
    axes[0, 1].set_xlabel("Pixels")
    
    # Difference
    diff_rgb = np.abs(after_snr[region, rgb_idx] - before_snr[region, rgb_idx])
    axes[0, 2].imshow(diff_rgb / 100)
    axes[0, 2].set_title("SNR Difference (RGB)", fontsize=11)
    axes[0, 2].set_xlabel("Pixels")
    
    # Noise distribution per band
    for band_idx in [0, 2, 4]:
        noise = after_snr[:, :, band_idx] - before_snr[:, :, band_idx]
        axes[1, band_idx // 2].hist(noise.ravel(), bins=50, edgecolor='black', alpha=0.7)
        axes[1, band_idx // 2].set_title(f"Band {band_idx} Noise Distribution", fontsize=11)
        axes[1, band_idx // 2].set_xlabel("Noise Value")
        axes[1, band_idx // 2].set_ylabel("Frequency")
    
    plt.tight_layout()
    plt.show()

<a name="psf"></a>
## 8. PSF Filtering

Apply Point Spread Function (PSF) filtering to simulate bandwidth limitations of the Module Transfer Function (MTF).

In [ ]:
# Apply PSF Filtering
psf_task = PhisatCalculationTask(
    input_feature=(FeatureType.DATA, current_feature),
    output_feature=(FeatureType.DATA, "BANDS_PSF"),
    executable=config.phisat2_exec_path,
    calculation="PSF",
)

if config.steps.psf_filtering:
    try:
        eopatch = psf_task.execute(eopatch)
        print("✓ PSF filtering applied")
    except Exception as e:
        print(f"⚠ PSF filtering failed: {e}")
        print("  (Check that phisat2 executable path is correct)")
        eopatch[FeatureType.DATA, "BANDS_PSF"] = eopatch[FeatureType.DATA, current_feature]
else:
    print("⊘ PSF filtering skipped")
    eopatch[FeatureType.DATA, "BANDS_PSF"] = eopatch[FeatureType.DATA, current_feature]

current_feature = "BANDS_PSF"

In [ ]:
# Visualize PSF effects (smoothing/blurring)
if "BANDS_PSF" in eopatch.data:
    before_psf = eopatch.data["BANDS_SNR"][0]
    after_psf = eopatch.data["BANDS_PSF"][0]
    
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    region = slice(150, 250), slice(150, 250)
    
    # RGB comparison
    axes[0, 0].imshow(np.clip(before_psf[region, rgb_idx] / 100, 0, 1))
    axes[0, 0].set_title("Before PSF", fontsize=11)
    axes[0, 0].set_xlabel("Pixels")
    
    axes[0, 1].imshow(np.clip(after_psf[region, rgb_idx] / 100, 0, 1))
    axes[0, 1].set_title("After PSF", fontsize=11)
    axes[0, 1].set_xlabel("Pixels")
    
    # Difference showing smoothing
    diff_rgb = np.abs(after_psf[region, rgb_idx] - before_psf[region, rgb_idx])
    axes[0, 2].imshow(diff_rgb / 100)
    axes[0, 2].set_title("PSF Difference", fontsize=11)
    axes[0, 2].set_xlabel("Pixels")
    
    # PSF effect statistics per band
    all_diffs = []
    for band_idx in range(after_psf.shape[2]):
        diff = np.abs(after_psf[:, :, band_idx] - before_psf[:, :, band_idx])
        all_diffs.append(diff.ravel())
    
    all_diffs_flat = np.concatenate(all_diffs)
    axes[1, 0].hist(all_diffs_flat, bins=50, edgecolor='black', log=True)
    axes[1, 0].set_title("PSF Difference Histogram (all bands)", fontsize=11)
    axes[1, 0].set_xlabel("Absolute Difference")
    axes[1, 0].set_ylabel("Frequency (log)")
    
    # Standard deviation per band
    stds = [np.std(d) for d in all_diffs]
    axes[1, 1].bar(range(len(stds)), stds)
    axes[1, 1].set_title("PSF Std Dev per Band", fontsize=11)
    axes[1, 1].set_xlabel("Band Index")
    axes[1, 1].set_ylabel("Std Dev")
    
    # Single band detail
    band_idx = 2
    axes[1, 2].imshow(before_psf[region, band_idx], cmap="gray", alpha=0.6, label="Before")
    axes[1, 2].imshow(after_psf[region, band_idx], cmap="hot", alpha=0.4, label="After")
    axes[1, 2].set_title(f"Band {band_idx} Overlay", fontsize=11)
    axes[1, 2].set_xlabel("Pixels")
    
    plt.tight_layout()
    plt.show()

<a name="reflectance"></a>
## 9. Reflectance Conversion

Convert radiances back to reflectances if processing level is L1C.

In [ ]:
# Apply Reflectance Conversion (only for L1C)
reflectance_task = CalculateReflectanceTask(
    (FeatureType.DATA, current_feature),
    (FeatureType.DATA, "PHISAT2_FINAL"),
    processing_level=processing_level,
)

if config.steps.reflectance_conversion and config.processing_level == "L1C":
    eopatch = reflectance_task.execute(eopatch)
    print("✓ Reflectance conversion applied (L1C)")
    final_feature = "PHISAT2_FINAL"
else:
    print(f"⊘ Reflectance conversion skipped ({config.processing_level} processing level)")
    # For L1A/L1B, radiance values are kept
    eopatch[FeatureType.DATA, "PHISAT2_FINAL"] = eopatch[FeatureType.DATA, current_feature]
    final_feature = "PHISAT2_FINAL"

print(f"\n✓ Simulation pipeline complete!")
print(f"  Final output shape: {eopatch.data[final_feature].shape}")
print(f"  Processing level: {config.processing_level}")

<a name="visualization"></a>
## 10. Comprehensive Results Visualization

Display the complete progression through all simulation steps and summary statistics.

In [ ]:
# Visualize complete workflow progression
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
region = slice(100, 300), slice(100, 300)
viz_factor = 3.5 if config.processing_level == "L1C" else 0.01

# 1. Original Reflectance
if "BANDS" in eopatch.data:
    axes[0, 0].imshow(np.clip(2.5 * eopatch.data["BANDS"][0, region[0], region[1], rgb_idx], 0, 1))
    axes[0, 0].set_title("1. Original Reflectance", fontsize=11, fontweight='bold')
    axes[0, 0].set_xlabel("Pixels")

# 2. After Radiance
if "BANDS-RAD" in eopatch.data:
    axes[0, 1].imshow(np.clip(eopatch.data["BANDS-RAD"][0, region[0], region[1], rgb_idx] / 100, 0, 1))
    axes[0, 1].set_title("2. Radiance Converted", fontsize=11, fontweight='bold')
    axes[0, 1].set_xlabel("Pixels")

# 3. After Panchromatic
if "BANDS-RAD-PAN_RES" in eopatch.data:
    axes[0, 2].imshow(np.clip(2.5 * eopatch.data["BANDS-RAD-PAN_RES"][0, region[0], region[1], rgb_idx], 0, 1))
    axes[0, 2].set_title("3. Panchromatic Added & Resampled", fontsize=11, fontweight='bold')
    axes[0, 2].set_xlabel("Pixels")

# 4. After Misalignment
if "BANDS_MISALIGNED" in eopatch.data:
    axes[1, 0].imshow(np.clip(2.5 * eopatch.data["BANDS_MISALIGNED"][0, region[0], region[1], rgb_idx], 0, 1))
    axes[1, 0].set_title("4. Band Misalignment Applied", fontsize=11, fontweight='bold')
    axes[1, 0].set_xlabel("Pixels")

# 5. After SNR + PSF
if "BANDS_PSF" in eopatch.data:
    axes[1, 1].imshow(np.clip(2.5 * eopatch.data["BANDS_PSF"][0, region[0], region[1], rgb_idx], 0, 1))
    axes[1, 1].set_title("5. SNR & PSF Applied", fontsize=11, fontweight='bold')
    axes[1, 1].set_xlabel("Pixels")

# 6. Final Output
if final_feature in eopatch.data:
    axes[1, 2].imshow(eopatch.data[final_feature][0, region[0], region[1], rgb_idx] * viz_factor)
    axes[1, 2].set_title(f"6. Final ({config.processing_level})", fontsize=11, fontweight='bold')
    axes[1, 2].set_xlabel("Pixels")

plt.suptitle("Φ-sat-2 Simulation Pipeline: Complete Workflow", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Summary Statistics and Analysis
print("\n" + "="*60)
print("SIMULATION PIPELINE SUMMARY")
print("="*60)

print(f"\nConfiguration:")
print(f"  Processing Level: {config.processing_level}")
print(f"  Output Directory: {config.output_dir}")
print(f"  SNR/PSF Method: {config.snr_psf_method}")

print(f"\nInput File:")
print(f"  Path: {s2_file}")
print(f"  Size: {s2_data.shape}")

print(f"\nProcessing Steps:")
for step, enabled in config.steps.as_dict().items():
    status = "✓ APPLIED" if enabled else "⊘ SKIPPED"
    print(f"  {status:13} {step.replace('_', ' ').title()}")

print(f"\nOutput Data:")
for feature_name in sorted(eopatch.data.keys()):
    shape = eopatch.data[feature_name].shape
    dtype = eopatch.data[feature_name].dtype
    min_val = eopatch.data[feature_name].min()
    max_val = eopatch.data[feature_name].max()
    print(f"  {feature_name:20} Shape: {shape} | Range: [{min_val:.2f}, {max_val:.2f}]")

print(f"\nEOPatch Summary:")
print(f"  {eopatch}")

print("\n" + "="*60)
print("✓ Simulation completed successfully!")
print("="*60)

In [ ]:
# Clean up intermediate features (based on simulation configuration)
# Build conditional list of features to remove based on simulation configuration
features_to_remove = [
    (FeatureType.DATA, "BANDS"),  # Always remove original S2 bands
]

# Add conditional removals based on enabled steps
if config.steps.radiance:
    features_to_remove.append((FeatureType.DATA, "BANDS-RAD"))

if config.steps.add_panchromatic:
    features_to_remove.extend([
        (FeatureType.DATA, "BANDS-RAD-PAN"),
        (FeatureType.DATA, "BANDS-RAD-PAN_RES"),
    ])

if config.steps.band_misalignment:
    features_to_remove.append((FeatureType.DATA, "BANDS_MISALIGNED"))

# Add features from SNR/PSF steps
if config.steps.snr_simulation:
    features_to_remove.append((FeatureType.DATA, "BANDS_SNR"))

if config.steps.psf_filtering:
    features_to_remove.append((FeatureType.DATA, "BANDS_PSF"))

# Only remove features that actually exist in the EOPatch
existing_features = [f for f in features_to_remove if f[1] in eopatch.data]

if existing_features:
    remove_task = RemoveFeatureTask(existing_features)
    eopatch = remove_task.execute(eopatch)
    print(f"✓ Removed {len(existing_features)} intermediate features")
    print(f"  Features removed: {[f[1] for f in existing_features]}")

print(f"\nFinal EOPatch after cleanup:")
print(f"  {eopatch}")
print(f"Final feature in EOPatch: {final_feature}")
print(f"Output shape: {eopatch.data[final_feature].shape}")

In [ ]:
output_dir = Path(config.output_dir) / "results"
output_dir.mkdir(parents=True, exist_ok=True)

export_task = ExportToTiffTask(
    feature=(FeatureType.DATA, final_feature),
    folder=str(output_dir),
)
export_task.execute(eopatch)
print(f"✓ Exported results to {output_dir}")